# Sentiment–Rating Incongruence in Sri Lanka Tourism Attraction Reviews
## Full Analysis Notebook — MERCon 2026

**Dataset:** 16,156 TripAdvisor attraction reviews (Mendeley Data, 2011–2023)  
**Sentiment:** cardiffnlp/twitter-roberta-base-sentiment (pre-computed)

---

| RQ | Question |
|----|----------|
| **RQ1** | How prevalent is incongruence, and what is the full typology of directional patterns? |
| **RQ2** | Does incongruence vary significantly across location types? |
| **RQ3** | Which reviewer-level and review-level characteristics independently predict incongruence? |


## 0. Imports & Configuration


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from matplotlib.patches import Patch
from scipy import stats
from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ────────────────────────────────────────────────────────────
P = dict(
    main  = '#4A6FA5',
    acc   = '#E05C5C',
    neu   = '#F0A500',
    grn   = '#2E9E6B',
    light = '#B8CCE4',
)

# ── Plot defaults ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#F9F9F7',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.color'       : '#E5E5E0',
    'grid.linewidth'   : 0.6,
    'font.family'      : 'sans-serif',
    'axes.titlesize'   : 12,
    'axes.titleweight' : '600',
    'axes.labelsize'   : 10,
    'xtick.labelsize'  : 9,
    'ytick.labelsize'  : 9,
    'legend.fontsize'  : 9,
})

print("✓  Libraries loaded.")


✓  Libraries loaded.


## 1. Load & Validate Dataset


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
df = pd.read_csv('Processed_Reviews_with_Sentiment.csv')

# ── Standardise Sentiment labels to title-case for consistent comparisons ────
df['Sentiment'] = df['Sentiment'].str.strip().str.upper()

# ── Validate Rating_Class coverage ───────────────────────────────────────────
assert set(df['Rating_Class'].unique()).issubset({'Positive','Neutral','Negative'}), \
    "Unexpected Rating_Class values"
assert set(df['Sentiment'].unique()).issubset({'POSITIVE','NEUTRAL','NEGATIVE'}), \
    "Unexpected Sentiment values"

print(f"✓  Dataset loaded  : {df.shape[0]:,} reviews × {df.shape[1]} columns")
print(f"   Year range      : {df['Travel_Date_Year'].min()} – {df['Travel_Date_Year'].max()}")
print(f"   Location types  : {df['Location_Type'].nunique()}")
print(f"   User countries  : {df['User_Country'].nunique()}")
print(f"\n   Rating_Class distribution:")
print(df['Rating_Class'].value_counts().to_string())
print(f"\n   Sentiment distribution:")
print(df['Sentiment'].value_counts().to_string())


✓  Dataset loaded  : 16,156 reviews × 21 columns
   Year range      : 2010 – 2023
   Location types  : 11
   User countries  : 150

   Rating_Class distribution:
Rating_Class
Positive    12845
Neutral      2166
Negative     1145

   Sentiment distribution:
Sentiment
POSITIVE    13041
NEGATIVE     1602
NEUTRAL      1513


## 2. Feature Engineering

> **Note:** `Rating_Class` and `Sentiment` are used **only** to construct the dependent variable `Incongruent`. They are never used as predictors — that would be circular.


In [7]:
# ─────────────────────────────────────────────────────────────────────────────

# ── 2a. Incongruent flag (dependent variable) ─────────────────────────────────
#   A review is CONGRUENT when Rating_Class and Sentiment agree:
#     Positive  ↔  POSITIVE
#     Neutral   ↔  NEUTRAL
#     Negative  ↔  NEGATIVE
#   All other combinations are INCONGRUENT.
#   NOTE: Rating and Sentiment are NEVER used as predictors — they define
#   the outcome and cannot appear on both sides of the model.

df['Incongruent'] = (~(
    ((df['Rating_Class'] == 'Positive') & (df['Sentiment'] == 'POSITIVE')) |
    ((df['Rating_Class'] == 'Neutral')  & (df['Sentiment'] == 'NEUTRAL'))  |
    ((df['Rating_Class'] == 'Negative') & (df['Sentiment'] == 'NEGATIVE'))
)).astype(int)

# ── 2b. Directional pattern — all six types ───────────────────────────────────
#   Each of the 6 possible incongruent combinations is named.
def assign_pattern(row):
    rc  = row['Rating_Class']   # Positive / Neutral / Negative
    sen = row['Sentiment']      # POSITIVE / NEUTRAL / NEGATIVE

    # Congruent first
    if   rc == 'Positive' and sen == 'POSITIVE': return 'Congruent–Positive'
    elif rc == 'Neutral'  and sen == 'NEUTRAL' : return 'Congruent–Neutral'
    elif rc == 'Negative' and sen == 'NEGATIVE': return 'Congruent–Negative'

    # Six incongruent patterns
    elif rc == 'Neutral'  and sen == 'POSITIVE': return 'Conservative Rater'
    elif rc == 'Positive' and sen == 'NEUTRAL' : return 'Obligatory 5-Star'
    elif rc == 'Neutral'  and sen == 'NEGATIVE': return 'Frustrated Neutral'
    elif rc == 'Positive' and sen == 'NEGATIVE': return 'Polite Inflator'
    elif rc == 'Negative' and sen == 'NEUTRAL' : return 'Harsh Deflator'
    elif rc == 'Negative' and sen == 'POSITIVE': return 'Punitive Rater'
    else: return 'Unknown'

df['Pattern'] = df.apply(assign_pattern, axis=1)

# ── 2c. Reviewer Tier (from User_Contributions) ───────────────────────────────
#   Cut-points follow the literature:
#     Novice   : 1 – 5   (first-time / occasional reviewers)
#     Casual   : 6 – 20  (regular contributors)
#     Active   : 21 – 100
#     Expert   : 101+    (power reviewers)

df['Reviewer_Tier'] = pd.cut(
    df['User_Contributions'],
    bins   = [0, 5, 20, 100, 100_000],
    labels = ['Novice (1–5)', 'Casual (6–20)', 'Active (21–100)', 'Expert (101+)'],
    right  = True
)

# ── 2d. Log-transform skewed continuous features ──────────────────────────────
#   Review_Length and Review_Delay_Days are right-skewed.
#   Log1p transformation stabilises variance for regression.
df['log_review_length'] = np.log1p(df['Review_Length'])
df['log_review_delay']  = np.log1p(df['Review_Delay_Days'])

# ── 2e. Verify no missing values in key columns ───────────────────────────────
key_cols = ['Incongruent','Pattern','Reviewer_Tier',
            'log_review_length','log_review_delay','Travel_Date_Year']
missing = df[key_cols].isnull().sum()
print("\n✓  Feature engineering complete.")
print(f"   Incongruent reviews : {df['Incongruent'].sum():,}  "
      f"({df['Incongruent'].mean()*100:.1f}%)")
print(f"\n   Reviewer Tier distribution:")
print(df['Reviewer_Tier'].value_counts().sort_index().to_string())
print(f"\n   Missing values in key columns:")
print(missing[missing > 0] if missing.sum() > 0 else "   None")



✓  Feature engineering complete.
   Incongruent reviews : 3,005  (18.6%)

   Reviewer Tier distribution:
Reviewer_Tier
Novice (1–5)       1293
Casual (6–20)      3186
Active (21–100)    6018
Expert (101+)      5659

   Missing values in key columns:
   None


## 3. RQ1 — Prevalence and Full Typology

**RQ1:** How prevalent is sentiment–rating incongruence, and what is the full typology of directional patterns?

| Pattern | Rating★ | Text | Mechanism |
|---------|---------|------|--------|
| Conservative Rater | Neutral 3★ | Positive | Rates cautiously despite positive experience |
| Obligatory 5-Star | Positive 4-5★ | Neutral | Social reciprocity; honest in text |
| Frustrated Neutral | Neutral 3★ | Negative | Softens rating despite negative experience |
| Polite Inflator | Positive 4-5★ | Negative | Generous rating despite negative experience |
| Harsh Deflator | Negative 1-2★ | Neutral | Overly critical rating; text is measured |
| Punitive Rater | Negative 1-2★ | Positive | Unexpectedly low rating despite positive text |


In [9]:
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  RQ1: PREVALENCE AND DIRECTIONAL TYPOLOGY")
print("="*65)

total     = len(df)
inc_count = df['Incongruent'].sum()
con_count = total - inc_count
inc_rate  = inc_count / total * 100

print(f"\n  Total reviews    : {total:,}")
print(f"  Congruent        : {con_count:,}  ({con_count/total*100:.1f}%)")
print(f"  Incongruent      : {inc_count:,}  ({inc_rate:.1f}%)")
print(f"  → ~1 in {int(round(1/df['Incongruent'].mean()))} reviews shows a mismatch.")

# ── Pattern breakdown ─────────────────────────────────────────────────────────
pattern_order = [
    'Conservative Rater',
    'Obligatory 5-Star',
    'Frustrated Neutral',
    'Polite Inflator',
    'Harsh Deflator',
    'Punitive Rater',
]

pattern_names = {
    'Conservative Rater' : 'Neutral★  |  Positive text  →  Conservative Rater',
    'Obligatory 5-Star'  : 'Positive★ |  Neutral text   →  Obligatory 5-Star',
    'Frustrated Neutral' : 'Neutral★  |  Negative text  →  Frustrated Neutral',
    'Polite Inflator'    : 'Positive★ |  Negative text  →  Polite Inflator',
    'Harsh Deflator'     : 'Negative★ |  Neutral text   →  Harsh Deflator',
    'Punitive Rater'     : 'Negative★ |  Positive text  →  Punitive Rater',
}

inc_df = df[df['Incongruent'] == 1]
pat_counts = inc_df['Pattern'].value_counts()

print(f"\n  Six incongruent directional patterns:\n")
print(f"  {'Pattern':<22}  {'n':>6}  {'% of Incongruent':>18}  {'% of All Reviews':>18}")
print(f"  {'-'*70}")
for p in pattern_order:
    n    = pat_counts.get(p, 0)
    p_i  = n / inc_count * 100
    p_a  = n / total    * 100
    print(f"  {p:<22}  {n:>6,}  {p_i:>17.1f}%  {p_a:>17.1f}%")
print(f"  {'-'*70}")
print(f"  {'TOTAL':<22}  {inc_count:>6,}  {100.0:>17.1f}%  {inc_rate:>17.1f}%")

# ── Figure 1: Prevalence doughnut + Typology bar ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Left: Doughnut
ax1 = axes[0]
ax1.set_facecolor('white')
wedge_colors = [P['main'], P['acc']]
wedges, _ = ax1.pie(
    [con_count, inc_count],
    colors      = wedge_colors,
    startangle  = 90,
    wedgeprops  = dict(width=0.52, edgecolor='white', linewidth=3)
)
ax1.text(0,  0.10, f'{inc_rate:.1f}%',
         ha='center', va='center', fontsize=24, fontweight='800', color=P['acc'])
ax1.text(0, -0.18, 'incongruent',
         ha='center', va='center', fontsize=11, color='#666')
ax1.text(0, -0.38, f'n = {inc_count:,} of {total:,}',
         ha='center', va='center', fontsize=9, color='#888')
ax1.legend(wedges, [f'Congruent ({con_count/total*100:.1f}%)',
                    f'Incongruent ({inc_rate:.1f}%)'],
           loc='lower center', bbox_to_anchor=(0.5, -0.06), frameon=False)
ax1.set_title('Overall Prevalence of Incongruence', pad=14, fontweight='700')

# Right: Horizontal bar — all 6 patterns
ax2   = axes[1]
pat_data  = [(p, pat_counts.get(p,0)) for p in pattern_order]
labels_r  = [f'{pattern_names[p].split("→")[1].strip()}' for p,_ in pat_data]
counts_r  = [c for _,c in pat_data]
pcts_r    = [c/inc_count*100 for c in counts_r]

bar_cols = [P['acc'], P['neu'], '#E8845C', '#C9506A', P['main'], P['grn']]
bars = ax2.barh(labels_r, counts_r,
                color=bar_cols, height=0.62, edgecolor='white', linewidth=0.8)

for bar, cnt, pct in zip(bars, counts_r, pcts_r):
    ax2.text(bar.get_width() + 12,
             bar.get_y() + bar.get_height()/2,
             f'{cnt:,}  ({pct:.1f}%)',
             va='center', fontsize=8.5, color='#333')

ax2.set_xlabel('Number of incongruent reviews')
ax2.set_xlim(0, max(counts_r) * 1.45)
ax2.set_title('Directional Typology of Incongruence\n(n = {:,} incongruent reviews)'.format(inc_count),
              pad=10, fontweight='700')

plt.tight_layout(pad=2.5)
plt.savefig('fig1_rq1_prevalence_typology.png', bbox_inches='tight', dpi=150)
plt.close()
print("\n  ✓  Figure 1 saved: fig1_rq1_prevalence_typology.png")



  RQ1: PREVALENCE AND DIRECTIONAL TYPOLOGY

  Total reviews    : 16,156
  Congruent        : 13,151  (81.4%)
  Incongruent      : 3,005  (18.6%)
  → ~1 in 5 reviews shows a mismatch.

  Six incongruent directional patterns:

  Pattern                      n    % of Incongruent    % of All Reviews
  ----------------------------------------------------------------------
  Conservative Rater       1,155               38.4%                7.1%
  Obligatory 5-Star          851               28.3%                5.3%
  Frustrated Neutral         509               16.9%                3.2%
  Polite Inflator            219                7.3%                1.4%
  Harsh Deflator             160                5.3%                1.0%
  Punitive Rater             111                3.7%                0.7%
  ----------------------------------------------------------------------
  TOTAL                    3,005              100.0%               18.6%



  ✓  Figure 1 saved: fig1_rq1_prevalence_typology.png


## 4. RQ2 — Incongruence by Location Type

**RQ2:** Does incongruence vary significantly across location types, and which venue types show the highest and lowest risk?

Reference category: **National Parks** (lowest incongruence rate).


In [11]:
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  RQ2: INCONGRUENCE ACROSS LOCATION TYPES")
print("="*65)

# ── 4a. Rates and Odds Ratios ─────────────────────────────────────────────────
lt_stats = df.groupby('Location_Type').agg(
    N           = ('Incongruent', 'count'),
    Incongruent = ('Incongruent', 'sum'),
    Rate        = ('Incongruent', 'mean')
).sort_values('Rate', ascending=False)

lt_stats['Rate_Pct'] = (lt_stats['Rate'] * 100).round(2)
lt_stats['Congruent']= lt_stats['N'] - lt_stats['Incongruent']
lt_stats['Odds']     = lt_stats['Incongruent'] / lt_stats['Congruent']

# Reference: National Parks (lowest rate)
ref_odds = lt_stats.loc['National Parks', 'Odds']
lt_stats['OR'] = (lt_stats['Odds'] / ref_odds).round(2)

print(f"\n  {'Location Type':<26} {'N':>6} {'Incongruent':>12} "
      f"{'Rate (%)':>10} {'OR vs NP':>10}")
print(f"  {'-'*68}")
for lt, row in lt_stats.iterrows():
    print(f"  {lt:<26} {row['N']:>6,} {row['Incongruent']:>12,} "
          f"{row['Rate_Pct']:>9.1f}% {row['OR']:>10.2f}")

# ── 4b. Chi-square test ───────────────────────────────────────────────────────
ct_lt              = pd.crosstab(df['Location_Type'], df['Incongruent'])
chi2_lt, p_lt, dof_lt, _ = chi2_contingency(ct_lt)
n_lt               = ct_lt.values.sum()
cramers_v          = np.sqrt(chi2_lt / (n_lt * (min(ct_lt.shape) - 1)))

print(f"\n  Chi-square test: Location Type × Incongruent")
print(f"  χ²({dof_lt}) = {chi2_lt:.2f},  p = {p_lt:.2e},  Cramér's V = {cramers_v:.4f}")
print(f"  → Significant variation in incongruence by location type.")

# ── 4c. Figure 2: Horizontal bar with OR labels ───────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))

lp    = lt_stats.sort_values('Rate_Pct')   # ascending for barh
rate_vals = lp['Rate_Pct'].values
mean_rate = df['Incongruent'].mean() * 100

col_list = []
for r in rate_vals:
    if r >= 23:    col_list.append(P['acc'])
    elif r >= 19:  col_list.append(P['neu'])
    else:          col_list.append(P['grn'])

bars = ax.barh(lp.index, rate_vals,
               color=col_list, height=0.65, edgecolor='white', linewidth=0.8)

for bar, rate, OR in zip(bars, rate_vals, lp['OR'].values):
    ax.text(bar.get_width() + 0.25,
            bar.get_y() + bar.get_height()/2,
            f'{rate:.1f}%   (OR = {OR:.2f}×)',
            va='center', fontsize=9, color='#333')

ax.axvline(mean_rate, color='#777', linestyle='--', linewidth=1.3, zorder=5)
ax.text(mean_rate + 0.2, -0.7,
        f'Overall mean\n({mean_rate:.1f}%)',
        fontsize=8, color='#777', va='top')

ax.set_xlabel('Incongruence Rate (%)')
ax.set_xlim(0, 37)
ax.set_title(
    f'Incongruence Rate by Location Type\n'
    f'χ²({dof_lt}) = {chi2_lt:.1f}, p < 0.001, Cramér\'s V = {cramers_v:.3f}'
    f'  |  Reference: National Parks',
    pad=10, fontweight='700')

legend_handles = [
    Patch(facecolor=P['acc'], label='High  (≥ 23%)'),
    Patch(facecolor=P['neu'], label='Medium (19–23%)'),
    Patch(facecolor=P['grn'], label='Low   (< 19%)'),
    mlines.Line2D([], [], color='#777', linestyle='--',
                  linewidth=1.3, label=f'Overall mean ({mean_rate:.1f}%)'),
]
ax.legend(handles=legend_handles, frameon=False, loc='lower right', fontsize=9)

plt.tight_layout(pad=2)
plt.savefig('fig2_rq2_location_type.png', bbox_inches='tight', dpi=150)
plt.close()
print("  ✓  Figure 2 saved: fig2_rq2_location_type.png")


# ── 4d. Pattern composition by location type ─────────────────────────────────
#   Shows which patterns dominate in high vs low incongruence venues.
print("\n  Pattern composition by location type (% of all reviews in that type):")
lt_pat = df.groupby(['Location_Type','Pattern']).size().unstack(fill_value=0)
lt_pat_pct = (lt_pat.div(lt_pat.sum(axis=1), axis=0) * 100).round(1)
for p in pattern_order:
    if p in lt_pat_pct.columns:
        lt_pat_pct[p] = lt_pat_pct[p]

display_cols = [c for c in pattern_order if c in lt_pat_pct.columns]
print(lt_pat_pct[display_cols].sort_values('Conservative Rater', ascending=False).to_string())



  RQ2: INCONGRUENCE ACROSS LOCATION TYPES

  Location Type                   N  Incongruent   Rate (%)   OR vs NP
  --------------------------------------------------------------------
  Museums                    1,525.0        401.0      26.3%       2.43
  Bodies of Water             839.0        199.0      23.7%       2.12
  Zoological Gardens          213.0         46.0      21.6%       1.88
  Religious Sites            3,017.0        596.0      19.8%       1.68
  Beaches                    2,110.0        401.0      19.0%       1.60
  Waterfalls                  933.0        168.0      18.0%       1.50
  Gardens                    1,354.0        235.0      17.4%       1.43
  Farms                      1,884.0        312.0      16.6%       1.35
  Nature & Wildlife Areas    1,557.0        256.0      16.4%       1.34
  Historic Sites             1,519.0        237.0      15.6%       1.26
  National Parks             1,205.0        154.0      12.8%       1.00

  Chi-square test: Locat

  ✓  Figure 2 saved: fig2_rq2_location_type.png

  Pattern composition by location type (% of all reviews in that type):
Pattern                  Conservative Rater  Obligatory 5-Star  Frustrated Neutral  Polite Inflator  Harsh Deflator  Punitive Rater
Location_Type                                                                                                                      
Bodies of Water                        11.6                5.4                 3.9              1.1             1.0             0.8
Zoological Gardens                     11.3                0.9                 5.2              2.8             0.5             0.9
Museums                                10.2                8.0                 2.6              3.6             0.8             1.0
Gardens                                10.0                1.2                 3.5              0.5             1.2             1.0
Farms                                   9.4                1.5                 3.1     

## 5. RQ3a — Bivariate Predictor Tests

Each candidate predictor is tested individually before logistic regression.

**Excluded from regression:**
- `Rating`/`Rating_Class`/`Sentiment` — define the DV (circular)
- `Helpful_Votes` — bivariate p = 0.54
- `Title_Length` — Δmedian = 1 char (trivial)
- Domestic flag — p = 0.96
- Season binary — p = 0.26
- `Location_Name`, `Located_City` — too granular


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  RQ3a — BIVARIATE PREDICTOR TESTS
#     Tests each candidate predictor individually before entering regression


In [13]:
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  RQ3a: BIVARIATE PREDICTOR TESTS")
print("="*65)

inc = df[df['Incongruent'] == 1]
con = df[df['Incongruent'] == 0]

# ── 5a. Reviewer Tier (chi-square) ────────────────────────────────────────────
ct_tier              = pd.crosstab(df['Reviewer_Tier'], df['Incongruent'])
chi2_ti, p_ti, _, _  = chi2_contingency(ct_tier)

tier_stats = df.groupby('Reviewer_Tier', observed=True).agg(
    N           = ('Incongruent','count'),
    Incongruent = ('Incongruent','sum'),
    Rate        = ('Incongruent','mean')
)
tier_stats['Rate_Pct'] = (tier_stats['Rate']*100).round(1)
tier_stats['Congruent']= tier_stats['N'] - tier_stats['Incongruent']
tier_stats['Odds']     = tier_stats['Incongruent'] / tier_stats['Congruent']
novice_odds            = tier_stats.loc['Novice (1–5)', 'Odds']
tier_stats['OR']       = (tier_stats['Odds'] / novice_odds).round(2)

print(f"\n  ── Reviewer Tier ─────────────────────────────")
print(tier_stats[['N','Incongruent','Rate_Pct','OR']].to_string())
print(f"  χ²(3) = {chi2_ti:.2f},  p = {p_ti:.2e}")
print(f"  Expert reviewers are {tier_stats.loc['Expert (101+)','OR']:.2f}× "
      f"more likely to be incongruent than Novices.")

# ── 5b. Review Length ─────────────────────────────────────────────────────────
u_len, p_len = mannwhitneyu(
    inc['log_review_length'], con['log_review_length'], alternative='two-sided')
print(f"\n  ── Review Length (log-transformed) ───────────")
print(f"  Median (incongruent) : {inc['Review_Length'].median():.0f} chars")
print(f"  Median (congruent)   : {con['Review_Length'].median():.0f} chars")
print(f"  Mann-Whitney U = {u_len:.0f},  p = {p_len:.4f}")
print(f"  → {'SIGNIFICANT' if p_len<0.05 else 'NOT significant'}")

# ── 5c. Review Delay ──────────────────────────────────────────────────────────
u_del, p_del = mannwhitneyu(
    inc['log_review_delay'], con['log_review_delay'], alternative='two-sided')
print(f"\n  ── Review Delay (log-transformed) ────────────")
print(f"  Median (incongruent) : {inc['Review_Delay_Days'].median():.0f} days")
print(f"  Median (congruent)   : {con['Review_Delay_Days'].median():.0f} days")
print(f"  Mann-Whitney U = {u_del:.0f},  p = {p_del:.4f}")
print(f"  → {'SIGNIFICANT' if p_del<0.05 else 'NOT significant'} "
      f"— emotional timing does NOT drive incongruence.")

# ── 5d. Province ──────────────────────────────────────────────────────────────
ct_prov              = pd.crosstab(df['Province'], df['Incongruent'])
chi2_pv, p_pv, dof_pv, _ = chi2_contingency(ct_prov)
prov_rates = df.groupby('Province')['Incongruent'].mean().mul(100).round(1).sort_values(ascending=False)
print(f"\n  ── Province ──────────────────────────────────")
print(f"  χ²({dof_pv}) = {chi2_pv:.2f},  p = {p_pv:.4f}")
print(f"  Province rates (%):")
print(prov_rates.to_string())

# ── 5e. Travel Year ───────────────────────────────────────────────────────────
u_yr, p_yr = mannwhitneyu(
    inc['Travel_Date_Year'], con['Travel_Date_Year'], alternative='two-sided')
print(f"\n  ── Travel Year ───────────────────────────────")
print(f"  Mann-Whitney U = {u_yr:.0f},  p = {p_yr:.4f}")
print(f"  → {'SIGNIFICANT' if p_yr<0.05 else 'NOT significant'} temporal drift.")

# ── 5f. Figure 3: Reviewer Tier + Review Length violin ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Tier bar
ax1   = axes[0]
ts    = tier_stats.sort_index()
tcols = [P['grn'], P['light'], P['main'], P['acc']]
b     = ax1.bar(ts.index.astype(str), ts['Rate_Pct'],
                color=tcols, edgecolor='white', width=0.6, linewidth=0.8)
for bar, rate, OR in zip(b, ts['Rate_Pct'], ts['OR']):
    ax1.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.35,
             f'{rate}%\nOR = {OR:.2f}×',
             ha='center', va='bottom', fontsize=8.5, fontweight='500')
ax1.axhline(mean_rate, color='#777', linestyle='--', linewidth=1.2,
            label=f'Overall mean ({mean_rate:.1f}%)')
ax1.set_ylim(0, 28)
ax1.set_xlabel('Reviewer Tier')
ax1.set_ylabel('Incongruence Rate (%)')
ax1.set_title(
    f'Incongruence by Reviewer Expertise\n'
    f'χ²(3) = {chi2_ti:.1f}, p < 0.001  |  Ref: Novice',
    pad=8, fontweight='700')
ax1.legend(frameon=False)

# Review Length violin
ax2      = axes[1]
cap_pct  = 0.97
cap_val  = df['Review_Length'].quantile(cap_pct)
plot_df  = df[df['Review_Length'] <= cap_val].copy()
parts = ax2.violinplot(
    [plot_df[plot_df['Incongruent']==0]['Review_Length'],
     plot_df[plot_df['Incongruent']==1]['Review_Length']],
    positions  = [0, 1],
    showmedians = True,
    showextrema = False
)
body_colors = [P['main'], P['acc']]
for pc, col in zip(parts['bodies'], body_colors):
    pc.set_facecolor(col); pc.set_alpha(0.68)
parts['cmedians'].set_color('white'); parts['cmedians'].set_linewidth(2)

ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Congruent', 'Incongruent'])
ax2.set_ylabel('Review Length (characters)')
ax2.set_title(
    f'Review Length by Congruence\n'
    f'Mann-Whitney p = {p_len:.4f}  |  '
    f'Median diff = +{int(inc["Review_Length"].median() - con["Review_Length"].median())} chars',
    pad=8, fontweight='700')

diff = int(inc['Review_Length'].median() - con['Review_Length'].median())
ax2.text(0.5, 0.93,
         f'Incongruent reviews are {diff} chars longer (median)',
         ha='center', transform=ax2.transAxes, fontsize=9, color='#444',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#f5f5f0', edgecolor='#ddd'))

plt.tight_layout(pad=2.5)
plt.savefig('fig3_rq3_reviewer_tier_length.png', bbox_inches='tight', dpi=150)
plt.close()
print("\n  ✓  Figure 3 saved: fig3_rq3_reviewer_tier_length.png")



  RQ3a: BIVARIATE PREDICTOR TESTS

  ── Reviewer Tier ─────────────────────────────
                    N  Incongruent  Rate_Pct    OR
Reviewer_Tier                                     
Novice (1–5)     1293          157      12.1  1.00
Casual (6–20)    3186          494      15.5  1.33
Active (21–100)  6018         1142      19.0  1.69
Expert (101+)    5659         1212      21.4  1.97
  χ²(3) = 85.99,  p = 1.59e-18
  Expert reviewers are 1.97× more likely to be incongruent than Novices.

  ── Review Length (log-transformed) ───────────
  Median (incongruent) : 296 chars
  Median (congruent)   : 279 chars
  Mann-Whitney U = 20513093,  p = 0.0011
  → SIGNIFICANT

  ── Review Delay (log-transformed) ────────────
  Median (incongruent) : 25 days
  Median (congruent)   : 25 days
  Mann-Whitney U = 19685986,  p = 0.7503
  → NOT significant — emotional timing does NOT drive incongruence.

  ── Province ──────────────────────────────────
  χ²(7) = 49.10,  p = 0.0000
  Province rates (%):
Pr


  ✓  Figure 3 saved: fig3_rq3_reviewer_tier_length.png


## 6. RQ3b — Logistic Regression

**DV:** `Incongruent` (0 = congruent, 1 = incongruent)

**Reference categories:** National Parks | Novice tier | Southern Province


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.  RQ3b — LOGISTIC REGRESSION (FULL PREDICTIVE MODEL)
#
#     Dependent variable : Incongruent (0 = congruent, 1 = incongruent)
#     Reference categories:
#       - Location_Type : National Parks
#       - Reviewer_Tier : Novice (1–5)
#       - Province      : Southern Province (lowest rate)
#     Excluded: Rating, Rating_Class, Sentiment (used to construct DV)
#               User_Contributions raw (subsumed by Reviewer_Tier)
#               Helpful_Votes (p = 0.54 bivariate, not significant)
#               Title_Length (p = 0.0001 bivariate but Δmedian = 1 char,
#                             trivial effect, excluded on practical grounds)
#               Domestic flag (p = 0.96, not significant)
#               Season (p = 0.26, not significant)
#               Location_Name, Located_City (too granular)
#               Published_Date_Year (correlated with Travel_Date_Year;
#                                    Travel year used as it captures
#                                    when experience occurred)


In [15]:
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  RQ3b: LOGISTIC REGRESSION")
print("="*65)

df_lr = df.dropna(subset=['Reviewer_Tier']).copy()
assert len(df_lr) == len(df), "Reviewer_Tier has unexpected NaNs — check cut bins"

# ── One-hot encode Location Type (drop National Parks as reference) ───────────
lt_dummies = pd.get_dummies(df_lr['Location_Type'], prefix='LT')
lt_ref     = 'LT_National Parks'
if lt_ref in lt_dummies.columns:
    lt_dummies = lt_dummies.drop(columns=[lt_ref])

# ── One-hot encode Province (drop Southern Province as reference) ─────────────
pv_dummies = pd.get_dummies(df_lr['Province'], prefix='PV')
pv_ref     = 'PV_Southern Province'
if pv_ref in pv_dummies.columns:
    pv_dummies = pv_dummies.drop(columns=[pv_ref])

# ── Ordinal encode Reviewer Tier (Novice = 0 as reference) ───────────────────
tier_map = {
    'Novice (1–5)'   : 0,
    'Casual (6–20)'  : 1,
    'Active (21–100)': 2,
    'Expert (101+)'  : 3,
}
df_lr['Tier_ord'] = df_lr['Reviewer_Tier'].map(tier_map)

# ── Assemble feature matrix ───────────────────────────────────────────────────
X = pd.concat([
    lt_dummies.reset_index(drop=True),
    pv_dummies.reset_index(drop=True),
    df_lr[['Tier_ord',
           'log_review_length',
           'log_review_delay',
           'Travel_Date_Year']].reset_index(drop=True),
], axis=1)
y = df_lr['Incongruent'].reset_index(drop=True)

print(f"\n  Model matrix: {X.shape[0]:,} observations × {X.shape[1]} predictors")
print(f"  Outcome rate: {y.mean()*100:.1f}% incongruent")
print(f"  Predictors included:")
for col in X.columns:
    print(f"    {col}")

# ── Scale features (for comparable coefficients & SE computation) ─────────────
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── Fit logistic regression ───────────────────────────────────────────────────
lr = LogisticRegression(max_iter=5000, random_state=42, C=1.0, solver='lbfgs')
lr.fit(X_scaled, y)

# ── Wald test standard errors (Hessian-based) ────────────────────────────────
p_hat = expit(X_scaled @ lr.coef_[0] + lr.intercept_[0])
W     = np.diag(p_hat * (1 - p_hat))
try:
    cov = np.linalg.inv(X_scaled.T @ W @ X_scaled)
except np.linalg.LinAlgError:
    # Fallback to pseudo-inverse if singular
    cov = np.linalg.pinv(X_scaled.T @ W @ X_scaled)

se   = np.sqrt(np.diag(cov))
z    = lr.coef_[0] / se
pval = 2 * (1 - stats.norm.cdf(np.abs(z)))

results = pd.DataFrame({
    'Predictor' : list(X.columns),
    'Coef'      : lr.coef_[0],
    'OR'        : np.exp(lr.coef_[0]),
    'CI_lo'     : np.exp(lr.coef_[0] - 1.96 * se),
    'CI_hi'     : np.exp(lr.coef_[0] + 1.96 * se),
    'SE'        : se,
    'Z'         : z,
    'p_value'   : pval,
})
results['Sig'] = results['p_value'].apply(
    lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns')))

results_sorted = results.sort_values('OR', ascending=False).reset_index(drop=True)

# ── AUC ───────────────────────────────────────────────────────────────────────
y_proba = lr.predict_proba(X_scaled)[:, 1]
auc     = roc_auc_score(y, y_proba)

print(f"\n  AUC-ROC = {auc:.4f}")
print(f"\n  Full regression results (sorted by OR):")
print(f"\n  {'Predictor':<35} {'Coef':>7} {'OR':>7} "
      f"{'95% CI':>18} {'p':>8} {'Sig':>5}")
print(f"  {'-'*82}")
for _, row in results_sorted.iterrows():
    ci = f"[{row['CI_lo']:.3f}, {row['CI_hi']:.3f}]"
    print(f"  {row['Predictor']:<35} {row['Coef']:>7.4f} {row['OR']:>7.4f} "
          f"{ci:>18} {row['p_value']:>8.4f} {row['Sig']:>5}")

sig_preds = results_sorted[results_sorted['p_value'] < 0.05]['Predictor'].tolist()
ns_preds  = results_sorted[results_sorted['p_value'] >= 0.05]['Predictor'].tolist()
print(f"\n  Significant predictors (p < 0.05): {len(sig_preds)}")
for p in sig_preds:
    print(f"    {p}")
print(f"\n  Non-significant predictors: {len(ns_preds)}")
for p in ns_preds:
    print(f"    {p}")


# ── Figure 4: Forest plot ─────────────────────────────────────────────────────
# Clean predictor labels for display
label_map = {
    'LT_Museums'              : 'Museums  (vs. Nat. Parks)',
    'LT_Bodies of Water'      : 'Bodies of Water  (vs. Nat. Parks)',
    'LT_Beaches'              : 'Beaches  (vs. Nat. Parks)',
    'LT_Religious Sites'      : 'Religious Sites  (vs. Nat. Parks)',
    'LT_Gardens'              : 'Gardens  (vs. Nat. Parks)',
    'LT_Waterfalls'           : 'Waterfalls  (vs. Nat. Parks)',
    'LT_Farms'                : 'Farms  (vs. Nat. Parks)',
    'LT_Historic Sites'       : 'Historic Sites  (vs. Nat. Parks)',
    'LT_Nature & Wildlife Areas': 'Nature & Wildlife Areas  (vs. Nat. Parks)',
    'LT_Zoological Gardens'   : 'Zoological Gardens  (vs. Nat. Parks)',
    'PV_Western Province'     : 'Western Province  (vs. Southern)',
    'PV_Northern Province'    : 'Northern Province  (vs. Southern)',
    'PV_North Central Province': 'North Central Province  (vs. Southern)',
    'PV_Central Province'     : 'Central Province  (vs. Southern)',
    'PV_Sabaragamuwa Province': 'Sabaragamuwa Province  (vs. Southern)',
    'PV_Uva Province'         : 'Uva Province  (vs. Southern)',
    'PV_Eastern Province'     : 'Eastern Province  (vs. Southern)',
    'Tier_ord'                : 'Reviewer Tier  (ordinal: Novice→Expert)',
    'log_review_length'       : 'Review Length  (log chars)',
    'log_review_delay'        : 'Review Delay  (log days)',
    'Travel_Date_Year'        : 'Travel Year',
}

rs = results_sorted.copy()
rs['label'] = rs['Predictor'].map(label_map).fillna(rs['Predictor'])

fig, ax = plt.subplots(figsize=(11, max(8, len(rs)*0.42)))

ypos = np.arange(len(rs))

for i, (_, row) in enumerate(rs.iterrows()):
    sig  = row['p_value'] < 0.05
    col  = P['acc'] if sig else '#AAAAAA'
    # CI bar
    ax.plot([row['CI_lo'], row['CI_hi']], [i, i],
            color=col, linewidth=1.6, zorder=2, solid_capstyle='round')
    ax.plot([row['CI_lo']]*2, [i-0.12, i+0.12], color=col, linewidth=1.6)
    ax.plot([row['CI_hi']]*2, [i-0.12, i+0.12], color=col, linewidth=1.6)
    # Point
    ax.scatter(row['OR'], i, color=col, s=65, zorder=3, edgecolors='white', linewidth=0.8)
    # Annotation
    ax.text(max(row['CI_hi'], row['OR']) + 0.015, i,
            f"{row['OR']:.3f}  {row['Sig']}",
            va='center', fontsize=7.8,
            color=P['acc'] if sig else '#999')

ax.axvline(1.0, color='#666', linestyle='--', linewidth=1.3, zorder=1)
ax.set_yticks(ypos)
ax.set_yticklabels(rs['label'], fontsize=8.2)
ax.set_xlabel('Odds Ratio  (with 95% Confidence Interval)', fontsize=10)
ax.set_title(
    f'Logistic Regression: Predictors of Sentiment–Rating Incongruence\n'
    f'AUC-ROC = {auc:.4f}  |  n = {len(y):,}  |  '
    f'Ref: National Parks, Novice tier, Southern Province',
    pad=10, fontweight='700')

h_sig = mlines.Line2D([], [], marker='o', color=P['acc'], markersize=8,
                       linestyle='none', label='Significant (p < 0.05)')
h_ns  = mlines.Line2D([], [], marker='o', color='#AAA', markersize=8,
                       linestyle='none', label='Not significant (p ≥ 0.05)')
h_ref = mlines.Line2D([], [], color='#666', linestyle='--',
                       linewidth=1.5, label='OR = 1.0 (reference)')
ax.legend(handles=[h_sig, h_ns, h_ref], frameon=False,
          loc='lower right', fontsize=9)

plt.tight_layout(pad=2)
plt.savefig('fig4_rq3_forest_plot.png', bbox_inches='tight', dpi=150)
plt.close()
print("\n  ✓  Figure 4 saved: fig4_rq3_forest_plot.png")


# ── Figure 5: ROC Curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5.5))
fpr, tpr, _ = roc_curve(y, y_proba)
ax.plot(fpr, tpr, color=P['acc'], linewidth=2.2,
        label=f'ROC curve  (AUC = {auc:.4f})')
ax.plot([0,1], [0,1], color='#bbb', linestyle='--',
        linewidth=1.3, label='Random classifier  (AUC = 0.500)')
ax.fill_between(fpr, tpr, alpha=0.07, color=P['acc'])
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Logistic Regression Model\n'
             '(Incongruence Prediction)', pad=8, fontweight='700')
ax.legend(frameon=False, fontsize=9)
ax.text(0.62, 0.18,
        f'AUC = {auc:.4f}\n'
        f'n = {len(y):,}  |  outcome rate = {y.mean()*100:.1f}%',
        transform=ax.transAxes, fontsize=9, color='#444',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#f5f5f0', edgecolor='#ccc'))
plt.tight_layout()
plt.savefig('fig5_roc_curve.png', bbox_inches='tight', dpi=150)
plt.close()
print("  ✓  Figure 5 saved: fig5_roc_curve.png")



  RQ3b: LOGISTIC REGRESSION

  Model matrix: 16,156 observations × 21 predictors
  Outcome rate: 18.6% incongruent
  Predictors included:
    LT_Beaches
    LT_Bodies of Water
    LT_Farms
    LT_Gardens
    LT_Historic Sites
    LT_Museums
    LT_Nature & Wildlife Areas
    LT_Religious Sites
    LT_Waterfalls
    LT_Zoological Gardens
    PV_Central Province
    PV_Eastern Province
    PV_North Central Province
    PV_Northern Province
    PV_Sabaragamuwa Province
    PV_Uva Province
    PV_Western Province
    Tier_ord
    log_review_length
    log_review_delay
    Travel_Date_Year



  AUC-ROC = 0.5983

  Full regression results (sorted by OR):

  Predictor                              Coef      OR             95% CI        p   Sig
  ----------------------------------------------------------------------------------
  LT_Beaches                           0.2594  1.2961     [1.202, 1.397]   0.0000   ***
  LT_Museums                           0.2548  1.2902     [1.207, 1.380]   0.0000   ***
  PV_Central Province                  0.2160  1.2411     [1.159, 1.329]   0.0000   ***
  PV_North Central Province            0.1881  1.2070     [1.127, 1.293]   0.0000   ***
  Tier_ord                             0.1641  1.1783     [1.129, 1.230]   0.0000   ***
  LT_Bodies of Water                   0.1543  1.1669     [1.105, 1.233]   0.0000   ***
  LT_Religious Sites                   0.1509  1.1629     [1.075, 1.259]   0.0002   ***
  log_review_length                    0.1299  1.1387     [1.092, 1.187]   0.0000   ***
  PV_Western Province                  0.1053  1.1111     [

## 7. Supplementary — Pattern × Reviewer Tier

Confirms the Conservative Rater pattern concentrates among Expert reviewers — the mechanistic link between RQ1 typology and the RQ3 expertise finding.


In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# 7.  SUPPLEMENTARY — PATTERN COMPOSITION BY REVIEWER TIER
#     Shows whether Expert reviewers skew toward Conservative Rater


In [17]:
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  SUPPLEMENTARY: PATTERN × REVIEWER TIER")
print("="*65)

tier_pat = (df[df['Incongruent']==1]
            .groupby(['Reviewer_Tier','Pattern'], observed=True)
            .size().unstack(fill_value=0))
tier_pat_pct = (tier_pat.div(tier_pat.sum(axis=1), axis=0) * 100).round(1)

print("\n  Pattern distribution within incongruent reviews by reviewer tier (%):")
disp_cols = [c for c in pattern_order if c in tier_pat_pct.columns]
print(tier_pat_pct[disp_cols].to_string())



  SUPPLEMENTARY: PATTERN × REVIEWER TIER

  Pattern distribution within incongruent reviews by reviewer tier (%):
Pattern          Conservative Rater  Obligatory 5-Star  Frustrated Neutral  Polite Inflator  Harsh Deflator  Punitive Rater
Reviewer_Tier                                                                                                              
Novice (1–5)                   27.4               28.7                19.7              7.6            10.8             5.7
Casual (6–20)                  37.2               25.1                16.6              8.5             7.1             5.5
Active (21–100)                38.4               27.6                17.5              7.9             5.0             3.7
Expert (101+)                  40.4               30.3                16.2              6.2             4.2             2.7


## 8. Full Results Summary


In [18]:
# ─────────────────────────────────────────────────────────────────────────────
# 8.  FULL RESULTS SUMMARY — ready to copy into paper


In [19]:
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*65)
print("  FULL RESULTS SUMMARY")
print("="*65)

print(f"""
── RQ1: Prevalence and Typology ─────────────────────────────
  Overall incongruence rate : {inc_rate:.1f}%  ({inc_count:,} / {total:,})
  ~1 in {int(round(1/df['Incongruent'].mean()))} reviews is incongruent.

  Six directional patterns:""")

for p in pattern_order:
    n   = pat_counts.get(p,0)
    pi  = n/inc_count*100
    pa  = n/total*100
    print(f"    {p:<22}  n={n:,}  ({pi:.1f}% of inc.  /  {pa:.1f}% of all)")

print(f"""
── RQ2: Location Type ───────────────────────────────────────
  χ²({dof_lt}) = {chi2_lt:.2f},  p < 0.001,  Cramér's V = {cramers_v:.4f}
  Highest : Museums        {lt_stats.loc['Museums','Rate_Pct']:.1f}%  (OR = {lt_stats.loc['Museums','OR']:.2f}×)
  Lowest  : National Parks {lt_stats.loc['National Parks','Rate_Pct']:.1f}%  (reference)

── RQ3: Predictors ──────────────────────────────────────────
  Reviewer Tier:  χ²(3) = {chi2_ti:.2f},  p < 0.001
    Expert vs Novice OR = {tier_stats.loc['Expert (101+)','OR']:.2f}×

  Review Length:  Mann-Whitney p = {p_len:.4f}  → SIGNIFICANT
    Median incongruent = {int(inc['Review_Length'].median())} chars
    Median congruent   = {int(con['Review_Length'].median())} chars

  Review Delay:   Mann-Whitney p = {p_del:.4f}  → NOT significant
    Emotional timing does NOT drive incongruence.

  Province:       χ²({dof_pv}) = {chi2_pv:.2f},  p = {p_pv:.4f}  → SIGNIFICANT

  Logistic Regression:
    AUC-ROC = {auc:.4f}
    Significant predictors (p < 0.05): {len(sig_preds)}
    Non-significant: {', '.join(ns_preds)}
""")



  FULL RESULTS SUMMARY

── RQ1: Prevalence and Typology ─────────────────────────────
  Overall incongruence rate : 18.6%  (3,005 / 16,156)
  ~1 in 5 reviews is incongruent.

  Six directional patterns:
    Conservative Rater      n=1,155  (38.4% of inc.  /  7.1% of all)
    Obligatory 5-Star       n=851  (28.3% of inc.  /  5.3% of all)
    Frustrated Neutral      n=509  (16.9% of inc.  /  3.2% of all)
    Polite Inflator         n=219  (7.3% of inc.  /  1.4% of all)
    Harsh Deflator          n=160  (5.3% of inc.  /  1.0% of all)
    Punitive Rater          n=111  (3.7% of inc.  /  0.7% of all)

── RQ2: Location Type ───────────────────────────────────────
  χ²(10) = 125.85,  p < 0.001,  Cramér's V = 0.0883
  Highest : Museums        26.3%  (OR = 2.43×)
  Lowest  : National Parks 12.8%  (reference)

── RQ3: Predictors ──────────────────────────────────────────
  Reviewer Tier:  χ²(3) = 85.99,  p < 0.001
    Expert vs Novice OR = 1.97×

  Review Length:  Mann-Whitney p = 0.0011  → SI

## 9. Output Verification


In [21]:
# ─────────────────────────────────────────────────────────────────────────────
import os
figures = [
    'fig1_rq1_prevalence_typology.png',
    'fig2_rq2_location_type.png',
    'fig3_rq3_reviewer_tier_length.png',
    'fig4_rq3_forest_plot.png',
    'fig5_roc_curve.png',
]
print("Output figures:")
for f in figures:
    status = '✓  SAVED' if os.path.exists(f) else '✗  MISSING'
    print(f"  [{status}]  {f}")

print("\n✓  Analysis complete.")


Output figures:
  [✓  SAVED]  fig1_rq1_prevalence_typology.png
  [✓  SAVED]  fig2_rq2_location_type.png
  [✓  SAVED]  fig3_rq3_reviewer_tier_length.png
  [✓  SAVED]  fig4_rq3_forest_plot.png
  [✓  SAVED]  fig5_roc_curve.png

✓  Analysis complete.
